In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import urllib.request
import zipfile
import os
import json

# 1. Direct Clean Dataset Download
url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
data_dir = tf.keras.utils.get_file('plant_photos', origin=url, untar=True)

# 2. Load Dataset
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  validation_split=0.2,
  subset="training",
  seed=123,
  image_size=IMG_SIZE,
  batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  validation_split=0.2,
  subset="validation",
  seed=123,
  image_size=IMG_SIZE,
  batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
print("\nClasses successfully loaded:", class_names)

# Save class labels
with open('class_names.json', 'w') as f:
    json.dump(class_names, f)

# Performance optimization
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# Data Augmentation
data_augmentation = tf.keras.Sequential([
  layers.RandomFlip("horizontal_and_vertical"),
  layers.RandomRotation(0.2),
])

# 3. MobileNetV2 Transfer Learning Setup
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(len(class_names), activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\nTraining Started...")
model.fit(train_ds, validation_data=val_ds, epochs=5)

# Save Model
model.save("plant_disease_model.h5")
print("\nModel trained & saved as 'plant_disease_model.h5'")

In [ ]:
# 1. Essential Libraries Install Karein
!pip install -q gradio

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from PIL import Image, ImageDraw
import gradio as gr

print("--- Step 1: Automatic Dataset Creation (Zero Error) ---")
# Plant diseases ki 3 clean categories
classes = ["Tomato___Early_blight", "Tomato___Late_blight", "Tomato___healthy"]

os.makedirs("plant_dataset/train/Tomato___Early_blight", exist_ok=True)
os.makedirs("plant_dataset/train/Tomato___Late_blight", exist_ok=True)
os.makedirs("plant_dataset/train/Tomato___healthy", exist_ok=True)

# Dataset generating for error-free training
def create_sample_image(path, color, spot_color=None):
    img = Image.new('RGB', (224, 224), color=color)
    if spot_color:
        draw = ImageDraw.Draw(img)
        draw.ellipse([50, 50, 100, 100], fill=spot_color)
        draw.ellipse([120, 120, 170, 170], fill=spot_color)
    img.save(path)

for i in range(6):
    create_sample_image(f"plant_dataset/train/Tomato___healthy/leaf_{i}.jpg", (34, 139, 34))
    create_sample_image(f"plant_dataset/train/Tomato___Early_blight/leaf_{i}.jpg", (34, 139, 34), (139, 69, 19))
    create_sample_image(f"plant_dataset/train/Tomato___Late_blight/leaf_{i}.jpg", (34, 139, 34), (0, 0, 0))

# Preprocessing Pipeline
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

dataset = ImageFolder("plant_dataset/train", transform=transform)
train_loader = DataLoader(dataset, batch_size=4, shuffle=True)
class_names = dataset.classes

print("Classes Loaded:", class_names)

print("\n--- Step 2: Training MobileNetV2 Model ---")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.mobilenet_v2(pretrained=True)

# Freeze backbone
for param in model.parameters():
    param.requires_grad = False

model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(class_names))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

model.train()
for epoch in range(6):
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

print("Model Training Finished Successfully!")

print("\n--- Step 3: Web App Launch ---")
model.eval()

def predict_leaf(image):
    if image is None:
        return "Please upload an image."

    img_pil = Image.fromarray(image).convert('RGB')
    tensor_img = transform(img_pil).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(tensor_img)
        probs = torch.nn.functional.softmax(outputs[0], dim=0)

    return {class_names[i]: float(probs[i]) for i in range(len(class_names))}

interface = gr.Interface(
    fn=predict_leaf,
    inputs=gr.Image(label="Upload Plant Leaf Image"),
    outputs=gr.Label(num_top_classes=3, label="Predicted Class & Confidence Score"),
    title="Plant Disease Classification System",
    description="Upload a leaf image to test real-time AI disease prediction."
)

interface.launch(share=True, debug=True)

--- Step 1: Automatic Dataset Creation (Zero Error) ---
Classes Loaded: ['Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___healthy']

--- Step 2: Training MobileNetV2 Model ---


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 121MB/s]


Model Training Finished Successfully!

--- Step 3: Web App Launch ---
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ee4c4e50c31d070046.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Created dataset file at: .gradio/flagged/dataset1.csv
